# 02 · pandas Fundamentals

**pandas** is the workhorse of Python data engineering: a `DataFrame` is a typed,
labeled table you can filter, transform and aggregate with concise code. This
notebook loads the raw files and covers selecting, filtering and deriving
columns.

In [ ]:
# ▶ Run this first. Locates the sample data no matter where the kernel starts.
from pathlib import Path

def find_data() -> Path:
    here = Path.cwd()
    for base in (here, *here.parents):
        if (base / 'data' / 'raw').exists():
            return base / 'data'
    raise FileNotFoundError('Run: uv run python data/build_data.py')

DATA = find_data()
RAW = DATA / 'raw'
print('Data directory:', DATA)
print('Raw files:', sorted(p.name for p in RAW.glob('*')))

## Reading data into a DataFrame

`read_csv` infers types and returns a `DataFrame`. `head`, `shape`, `dtypes` and
`info` are your first look at any dataset.

In [ ]:
import pandas as pd

orders = pd.read_csv(RAW / 'orders.csv', parse_dates=['order_ts'])
print('shape:', orders.shape)
print('dtypes:\n', orders.dtypes)
orders.head()

## Series vs DataFrame

A **column** is a `Series` (a labeled 1-D array, NumPy underneath). A
**DataFrame** is a dict-like collection of aligned Series sharing an index.

In [ ]:
amounts = orders['amount']          # a Series
print(type(amounts))
print(amounts.describe())           # count/mean/std/min/quartiles/max

## Selecting columns and rows

Select columns by name; select rows by boolean condition (the pandas equivalent
of the NumPy masks from notebook 01). `.loc` selects by label, `.iloc` by
position.

In [ ]:
# columns
print(orders[['order_id', 'amount']].head(3))
print('---')
# rows by position
print(orders.iloc[0])
print('--- big completed orders ---')
# rows by condition
big = orders[(orders['amount'] >= 200) & (orders['status'] == 'completed')]
print(big[['order_id', 'amount', 'status']].head())

## Deriving new columns (vectorized)

Assign a new column from an expression over existing ones — applied to the whole
column at once, no loop. `np.where` and `.dt`/`.str` accessors handle
conditionals, dates and text.

In [ ]:
import numpy as np

orders['amount_tier'] = np.where(orders['amount'] >= 200, 'big', 'small')
orders['order_month'] = orders['order_ts'].dt.to_period('M').astype(str)
orders['is_completed'] = orders['status'].eq('completed')
print(orders[['order_id', 'amount', 'amount_tier', 'order_month', 'is_completed']].head())

## Quick aggregations

Series methods give instant summaries; `value_counts` tallies categories.

In [ ]:
print('total revenue:', round(orders.loc[orders.is_completed, 'amount'].sum(), 2))
print('avg order:', round(orders['amount'].mean(), 2))
print('\nstatus counts:')
print(orders['status'].value_counts())

## Reading from SQL directly

`pandas.read_sql` runs a query against a SQLAlchemy engine and returns a
DataFrame — the bridge between the database module and analysis.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine(f'sqlite:///{DATA / "retail.db"}')
products = pd.read_sql('SELECT * FROM products', engine)
print(products.shape)
products.head()

## `.loc` vs `.iloc` — label vs position

Two explicit selectors you'll use constantly:

- **`.loc[rows, cols]`** selects by **label** (index values and column names)
  and is **inclusive** of the end in a slice.
- **`.iloc[rows, cols]`** selects by **integer position** (0-based, end
  exclusive, like normal Python slicing).

Being explicit avoids the ambiguity of chained `[]` indexing.

In [ ]:
# .iloc — by position
print('first row, first 3 cols:')
print(orders.iloc[0, :3])
print('rows 0-2, columns 0 and 4:')
print(orders.iloc[0:3, [0, 4]])

# .loc — by label + boolean mask, selecting specific columns
print('\nbig completed orders (label-based):')
print(orders.loc[orders['amount'] >= 200, ['order_id', 'amount', 'status']].head(3))

## Categorical dtype — memory & speed

A column with few distinct values (status, country, category) stored as
`category` uses far less memory and speeds up grouping — pandas stores each label
once and keeps small integer codes per row. A key optimization on large tables.

In [ ]:
before = orders['status'].memory_usage(deep=True)
orders['status'] = orders['status'].astype('category')
after = orders['status'].memory_usage(deep=True)
print('dtype now:', orders['status'].dtype)
print('categories:', list(orders['status'].cat.categories))
print(f'memory: {before} -> {after} bytes')

### Recap

`read_csv`/`read_sql` load tables; a column is a `Series`, a table a
`DataFrame`; select columns by name and rows by boolean mask; `.loc` is
label-based (end-inclusive), `.iloc` position-based (end-exclusive); derive
columns vectorized with `np.where` and `.dt`/`.str`; `describe`/`value_counts`
summarize; `category` dtype saves memory on low-cardinality columns. Next:
grouping, joining and reshaping.